<a href="https://colab.research.google.com/github/stathis-liap/RL-Humanoid-Walking/blob/main/humanoid_RL_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install MuJoCo and RL libraries
!pip install gymnasium[mujoco] stable-baselines3[extra]

# 2. Upload your ZIP file
from google.colab import files
import zipfile
import os

uploaded = files.upload()

# 3. Extract the project
for fn in uploaded.keys():
    with zipfile.ZipFile(fn, 'r') as zip_ref:
        zip_ref.extractall(".")
    print(f"Extracted {fn} successfully!")

# 4. Fix permissions (Linux requirement)
!chmod -R 777 /content/project

In [2]:
%cd /content/project
%ls -la

/content/project
total 40
drwxrwxrwx 6 root root 4096 Mar 20 15:41 ./
drwxr-xr-x 1 root root 4096 Mar 20 15:36 ../
drwxrwxrwx 3 root root 4096 Mar 20 15:36 humanoid_lite/
-rwxrwxrwx 1 root root 5261 Mar 20 15:41 humanoid_lite_env.py*
drwxr-xr-x 2 root root 4096 Mar 20 15:43 models/
drwxr-xr-x 2 root root 4096 Mar 20 15:41 __pycache__/
drwxr-xr-x 3 root root 4096 Mar 20 15:41 tb_logs/
-rwxrwxrwx 1 root root 4201 Mar 20 15:39 train_humanoid_lite.py*


In [ ]:
!apt-get update
!apt-get install -y xvfb x11-utils
!pip install pyvirtualdisplay

In [4]:
%%writefile train_humanoid_lite.py
import os
import re

# --- HEADLESS COLAB DISPLAY FIX ---
# This MUST come before importing gymnasium or mujoco!
os.environ["MUJOCO_GL"] = "egl" # Tells MuJoCo to use the GPU for rendering
from pyvirtualdisplay import Display
display = Display(visible=False, size=(640, 480))
display.start()
# ----------------------------------

import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback, CallbackList
from gymnasium.wrappers import RecordEpisodeStatistics
from gymnasium.utils.save_video import save_video

from humanoid_lite_env import HumanoidLiteEnv

CHECKPOINT_DIR = "./models/"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
VEC_NORM_PATH = os.path.join(CHECKPOINT_DIR, "vecnormalize.pkl")

def make_train_env():
    env = HumanoidLiteEnv(render_mode="rgb_array")
    env = RecordEpisodeStatistics(env)
    return env

train_env = DummyVecEnv([make_train_env])
eval_env = DummyVecEnv([lambda: HumanoidLiteEnv(render_mode="rgb_array")])

checkpoints = [
    f for f in os.listdir(CHECKPOINT_DIR)
    if f.startswith("ppo_humanoid_lite_") and f.endswith("_steps.zip")
]

def extract_steps(fname: str) -> int:
    m = re.search(r"ppo_humanoid_lite_(\d+)_steps\.zip", fname)
    return int(m.group(1)) if m else -1

TARGET_TOTAL_STEPS = 50_000

if checkpoints and os.path.exists(VEC_NORM_PATH):
    latest = max(checkpoints, key=extract_steps)
    last_step = extract_steps(latest)
    print(f"Resuming from {latest} (step {last_step:,})")

    train_env = VecNormalize.load(VEC_NORM_PATH, train_env)
    train_env.training = True
    train_env.norm_reward = True

    eval_env = VecNormalize.load(VEC_NORM_PATH, eval_env)
    eval_env.training = False
    eval_env.norm_reward = False

    model = PPO.load(
        os.path.join(CHECKPOINT_DIR, latest),
        env=train_env,
        device="auto",
        custom_objects={"learning_rate": 1e-6, "clip_range": 0.1}
    )
    start_step = last_step
else:
    print("Starting brand-new training.")
    train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True, clip_obs=10.0)
    eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, clip_obs=10.0)

    model = PPO(
        "MlpPolicy", train_env, verbose=1, learning_rate=3e-4, n_steps=2048,
        batch_size=256, gamma=0.99, gae_lambda=0.95, clip_range=0.2,
        ent_coef=0.005, target_kl=0.03, tensorboard_log="./tb_logs/", device="auto"
    )
    start_step = 0

checkpoint_cb = CheckpointCallback(save_freq=10_000, save_path=CHECKPOINT_DIR, name_prefix="ppo_humanoid_lite")
eval_cb = EvalCallback(eval_env, best_model_save_path=CHECKPOINT_DIR, log_path=CHECKPOINT_DIR, eval_freq=10_000, n_eval_episodes=5, deterministic=True, render=False)
callbacks = CallbackList([checkpoint_cb, eval_cb])

remaining = TARGET_TOTAL_STEPS - start_step
if remaining > 0:
    print(f"Training {remaining:,} steps → total {TARGET_TOTAL_STEPS:,}")
    try:
        model.learn(total_timesteps=remaining, callback=callbacks, reset_num_timesteps=False)
    except KeyboardInterrupt:
        print("Training interrupted manually. Saving...")

final_path = os.path.join(CHECKPOINT_DIR, f"humanoid_lite_ppo_{TARGET_TOTAL_STEPS}_steps.zip")
model.save(final_path)
train_env.save(VEC_NORM_PATH)
print(f"Final model → {final_path}")

print("\nGenerating Live Demo Video for Colab...")
video_env = HumanoidLiteEnv(render_mode="rgb_array")
video_vec_env = DummyVecEnv([lambda: video_env])
video_vec_env = VecNormalize.load(VEC_NORM_PATH, video_vec_env)
video_vec_env.training = False
video_vec_env.norm_reward = False

obs = video_vec_env.reset()
frames = []

for _ in range(500):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = video_vec_env.step(action)
    frames.append(video_env.render())
    if done:
        obs = video_vec_env.reset()

video_vec_env.close()

video_dir = os.path.join(CHECKPOINT_DIR, "videos")
os.makedirs(video_dir, exist_ok=True)
save_video(frames, video_dir, fps=60, name_prefix="final_humanoid_stand")
print(f"Demo video saved to {video_dir}")

Overwriting train_humanoid_lite.py


In [7]:
%%writefile humanoid_lite_env.py
import os
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import mujoco

class HumanoidLiteEnv(gym.Env):
    metadata = {"render_modes": ["rgb_array", "human"], "render_fps": 60}

    def __init__(self, render_mode="rgb_array"):
        super().__init__()

        # Colab path safety: assuming your xml is uploaded to /content/humanoid_lite/bhl_scene.xml
        xml_path = "/content/project/humanoid_lite/bhl_scene.xml"

        if not os.path.exists(xml_path):
            raise FileNotFoundError(f"Could not find XML at {xml_path}. Make sure the humanoid_lite folder is uploaded to /content/")

        self.model = mujoco.MjModel.from_xml_path(xml_path)
        self.data = mujoco.MjData(self.model)

        self.render_mode = render_mode
        self.renderer = None
        if self.render_mode == "rgb_array":
            # Native offscreen renderer for Colab video recording
            self.renderer = mujoco.Renderer(self.model, height=480, width=640)

        # Action space: joints
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(self.model.nu,), dtype=np.float32)

        # Observation space: qpos (minus X and Y) + qvel
        obs_dim = (self.model.nq - 2) + self.model.nv
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)

        # Curriculum / Push Variables
        self._current_step = 0
        self._push_active = False
        self._push_steps_left = 0
        self._push_force = np.zeros(3, dtype=np.float64)
        self._next_push_step = 500

    def _get_obs(self):
        # Skip qpos[0] and qpos[1] (global X and Y) to prevent location overfitting
        qpos_clean = self.data.qpos[2:].copy()
        qvel_clean = self.data.qvel.copy()
        return np.concatenate([qpos_clean, qvel_clean]).astype(np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetDataKeyframe(self.model, self.data, 0)
        self.data.qpos[:] = 0.0

        # Add slight random noise to initial posture to prevent memorization
        self.data.qpos[2] = 0.61 + np.random.uniform(-0.02, 0.02) # Z height
        self.data.qpos[7:] += np.random.uniform(-0.05, 0.05, size=self.model.nq - 7)
        self.data.qvel[:] = np.random.uniform(-0.01, 0.01, size=self.model.nv)

        mujoco.mj_forward(self.model, self.data)

        self._current_step = 0
        self._push_active = False
        self._next_push_step = np.random.randint(400, 800)

        return self._get_obs(), {}

    def step(self, action):
        self._current_step += 1

        # Control mapping
        action = np.clip(action, -1.0, 1.0)
        self.data.ctrl[:] = 20.0 * action

        # Push Curriculum (Intermittently push the robot to teach balance recovery)
        if not self._push_active and self._current_step >= self._next_push_step:
            self._push_active = True
            self._push_steps_left = 10
            self._push_force[:] = np.random.uniform(-10.0, 10.0, size=3)
            self._push_force[2] = np.random.uniform(-2.0, 2.0)
            self._next_push_step = self._current_step + np.random.randint(400, 800)

        if self._push_active:
            base_id = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY, "imu_2")
            mujoco.mj_applyFT(
                self.model, self.data, self._push_force, np.zeros(3), np.zeros(3),
                base_id, self.data.qfrc_applied
            )
            self._push_steps_left -= 1
            if self._push_steps_left <= 0:
                self._push_active = False

        # Physics Step
        mujoco.mj_step(self.model, self.data)

        # Reward Calculation
        base_id = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY, "imu_2")
        torso_pos = self.data.xpos[base_id]
        torso_z_axis = self.data.xmat[base_id].reshape(3, 3)[:, 2]

        torso_height = torso_pos[2]
        upright_cos = torso_z_axis[2]

        # 1. Alive Bonus
        alive_reward = 1.0
        # 2. Posture Reward (Exponential drop-off as it leans)
        upright_reward = np.exp(-4.0 * (1.0 - upright_cos)**2)
        # 3. Height Reward (Targeting ~0.61m standing height)
        height_reward = np.exp(-20.0 * (0.61 - torso_height)**2)

        # 4. Penalties
        lin_vel = np.linalg.norm(self.data.qvel[:3])
        ang_vel = np.linalg.norm(self.data.qvel[3:6])
        vel_penalty = 0.5 * np.exp(-lin_vel) + 0.5 * np.exp(-ang_vel)
        ctrl_penalty = 0.05 * np.sum(np.square(action))

        # Total Reward
        reward = 1.0 * alive_reward + 2.0 * upright_reward + 2.0 * height_reward + 1.0 * vel_penalty - ctrl_penalty

        # Termination Condition (Fallen over)
        terminated = bool(torso_height < 0.35 or upright_cos < 0.2)

        return self._get_obs(), float(reward), terminated, False, {}

    def render(self):
        if self.render_mode == "rgb_array" and self.renderer is not None:
            self.renderer.update_scene(self.data)
            return self.renderer.render()
        return None

    def close(self):
        if self.renderer is not None:
            self.renderer.close()

Overwriting humanoid_lite_env.py


In [ ]:
!python train_humanoid_lite.py

In [3]:
import os
# We ONLY need EGL. No pyvirtualdisplay needed for rgb_array!
os.environ["MUJOCO_GL"] = "egl"

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from gymnasium.utils.save_video import save_video
from IPython.display import HTML, display
from base64 import b64encode
from tqdm import tqdm  # This gives us a progress bar

from humanoid_lite_env import HumanoidLiteEnv

CHECKPOINT_DIR = "./models/"
VEC_NORM_PATH = os.path.join(CHECKPOINT_DIR, "vecnormalize.pkl")
MODEL_PATH = os.path.join(CHECKPOINT_DIR, "humanoid_lite_ppo_50000_steps.zip")

print("Loading environment and model...")
video_env = HumanoidLiteEnv(render_mode="rgb_array")
video_vec_env = DummyVecEnv([lambda: video_env])
video_vec_env = VecNormalize.load(VEC_NORM_PATH, video_vec_env)
video_vec_env.training = False
video_vec_env.norm_reward = False

# CRITICAL FIX: Force PyTorch to use the CPU during evaluation so it doesn't fight MuJoCo for the GPU
model = PPO.load(MODEL_PATH, device="cpu")

print("Recording 500 frames (~8 seconds)...")
obs = video_vec_env.reset()
frames = []

# Wrap the range in tqdm to see the progress!
for _ in tqdm(range(500), desc="Rendering frames"):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = video_vec_env.step(action)
    frames.append(video_env.render())

video_vec_env.close()

# Save the video
video_dir = os.path.join(CHECKPOINT_DIR, "videos")
os.makedirs(video_dir, exist_ok=True)
save_video(frames, video_dir, fps=60, name_prefix="final_humanoid_stand")
print(f"\nDemo video saved to {video_dir}")

# Display it right here in Colab
mp4_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
if mp4_files:
    mp4_path = os.path.join(video_dir, sorted(mp4_files)[-1])
    mp4 = open(mp4_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

    display(HTML(f"""
    <video width=640 controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    """))

Loading environment and model...
Recording 500 frames (~8 seconds)...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
Rendering frames: 100%|██████████| 500/500 [16:38<00:00,  2.00s/it]



Demo video saved to ./models/videos
